[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap25_sac_price_comprar_alugar.ipynb)

# Capítulo 25: Quero uma casa: comprar, alugar ou financiar

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Você compara as tabelas SAC e Price de um financiamento e testa a conta de comprar ou alugar e investir a diferença.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. SAC x Price
Exemplo simplificado do capítulo 25, sem TR e sem seguros.

In [ ]:
FINANCIADO = 400_000   # MUDE AQUI
JUROS_ANO = 0.113      # MUDE AQUI
MESES = 360            # MUDE AQUI
i = (1 + JUROS_ANO) ** (1/12) - 1
price = FINANCIADO * i / (1 - (1 + i) ** -MESES)
saldo = FINANCIADO; sac = []
amort = FINANCIADO / MESES
for k in range(MESES):
    sac.append(amort + saldo * i); saldo -= amort
print(f'Price: parcela fixa de {brl(price)}; juros totais {brl(price*MESES - FINANCIADO)}; total pago {brl(price*MESES)}')
print(f'SAC: primeira parcela {brl(sac[0])}, última {brl(sac[-1])}; juros totais {brl(sum(sac) - FINANCIADO)}; total pago {brl(sum(sac))}')
fig, ax = plt.subplots()
ax.plot(np.arange(1, MESES+1)/12, [price]*MESES, color=CORES['laranja'], lw=2, label='Price')
ax.plot(np.arange(1, MESES+1)/12, sac, color=CORES['azul'], lw=2, label='SAC')
ax.set_xlabel('Anos'); ax.set_title('Valor da parcela'); eixo_reais(ax); ax.legend(); plt.show()

## 2. Comprar ou alugar e investir
Conta simplificada, em valores de hoje: quem aluga investe a entrada e a diferença entre a parcela e o aluguel. Valorização do imóvel e rendimento são hipóteses suas.

In [ ]:
IMOVEL = 500_000          # MUDE AQUI
ENTRADA = 100_000         # MUDE AQUI
ALUGUEL = 2_200           # MUDE AQUI: aluguel mensal de um imóvel parecido
VALORIZACAO_REAL = 0.00   # MUDE AQUI: valorização do imóvel acima da inflação ao ano
RENDIMENTO_REAL = 0.04    # MUDE AQUI: rendimento dos investimentos acima da inflação ao ano
INFLACAO = 0.04           # MUDE AQUI: para trazer as parcelas nominais a valores de hoje
ANOS = 30
ir = (1 + RENDIMENTO_REAL) ** (1/12) - 1
pi = (1 + INFLACAO) ** (1/12) - 1
invest = ENTRADA
for k in range(ANOS*12):
    parcela_real = price / (1 + pi) ** k     # Price: parcela nominal fixa perde valor real
    invest = invest * (1 + ir) + max(0, parcela_real - ALUGUEL)
imovel_final = IMOVEL * (1 + VALORIZACAO_REAL) ** ANOS
print(f'Quem comprou: imóvel quitado de cerca de {brl(imovel_final)} (em reais de hoje)')
print(f'Quem alugou e investiu a diferença: cerca de {brl(invest)} (em reais de hoje)')

A conta muda muito com o aluguel, os juros, a valorização e a disciplina para investir a diferença. Os custos de compra (ITBI, cartório) estão no capítulo 25. Simule com os seus números na [calculadora de financiamento SAC, Price e comprar ou alugar](https://www.traderbrasil.com/blog/calculadora-financiamento-sac-price-comprar-ou-alugar.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 25 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap25/

Achou um erro? Escreva para contato@traderbrasil.com.